# 16. The Storage Location Assignment Problem (SLAP)
## Tier 3: The Advanced Algorithm - Moth-Flame Optimization

### Key assumptions
- Moths represent potential storage assignments flying in search of optimal solutions
- Flames represent high-quality solutions that guide the search process
- Spiral flight patterns balance exploration and exploitation
- Number of flames decreases over iterations to force convergence

### Approach (step-by-step)
1. **Population Initialization**: Generate random moth assignments as initial solutions
2. **Fitness Evaluation**: Calculate fitness (inverse of total cost) for each moth
3. **Flame Selection**: Identify best moths as flames to guide the search
4. **Spiral Update**: Moths fly in spiral paths toward flames using logarithmic spirals
5. **Adaptive Flames**: Reduce number of flames over iterations for convergence
6. **Best Solution**: Track and return the best assignment found

### What to look for in the results
- Convergence behavior showing cost reduction over iterations
- Spiral flight patterns visualizing moth movement toward optimal solutions
- Final assignment with significant improvement over random and greedy baselines
- Comparison showing 23% improvement over random assignment and 8% over greedy heuristic

### Concrete example (from the source)
We'll implement the example with 6 products and 4 locations:
- Products with velocities: [45, 30, 25, 18, 12, 8]
- Locations with distances: [2.0, 4.5, 6.0, 8.5]
- Population size: 20 moths, iterations: 150
- Expected final assignment: [0, 0, 1, 1, 2, 2]
- Expected total cost: $384.50 with 23% improvement over random assignment

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict
import random
import time
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Product:
    """Represents a product/SKU in the warehouse"""
    id: int
    velocity: float  # Demand frequency
    space_req: float  # Space requirement
    category: int = 0  # Product category for affinity constraints
    name: str = ""

@dataclass
class Location:
    """Represents a storage location"""
    id: str
    distance: float  # Distance from I/O point
    capacity: float  # Storage capacity
    zone: int = 0  # Storage zone

@dataclass
class Moth:
    """Represents a moth (candidate solution) in MFO"""
    position: np.ndarray  # Assignment vector
    fitness: float  # Fitness value (inverse of cost)
    cost: float  # Total assignment cost

@dataclass
class SLAPInstance:
    """Storage Location Assignment Problem instance"""
    products: List[Product]
    locations: List[Location]
    affinity_matrix: np.ndarray  # Product affinity scores
    max_distance: float = 2.0  # Maximum distance for affinity constraints

In [ ]:
def calculate_assignment_cost(instance: SLAPInstance, assignment: np.ndarray) -> float:
    """
    Calculate total cost for a given assignment.
    Cost = distance * velocity + affinity penalties
    """
    total_cost = 0.0
    n_products = len(instance.products)
    n_locations = len(instance.locations)
    
    # Base assignment costs
    for i in range(n_products):
        loc_idx = int(assignment[i]) % n_locations
        total_cost += instance.locations[loc_idx].distance * instance.products[i].velocity
    
    # Affinity penalties for separating related products
    for i in range(n_products):
        for j in range(i + 1, n_products):
            loc_i = int(assignment[i]) % n_locations
            loc_j = int(assignment[j]) % n_locations
            
            # Distance between locations
            location_distance = abs(instance.locations[loc_i].distance - instance.locations[loc_j].distance)
            
            # Penalty if related products are far apart
            if instance.affinity_matrix[i, j] > 0 and location_distance > instance.max_distance:
                penalty = instance.affinity_matrix[i, j] * (location_distance - instance.max_distance)
                total_cost += penalty
    
    return total_cost

def check_feasibility(instance: SLAPInstance, assignment: np.ndarray) -> bool:
    """
    Check if assignment respects capacity constraints.
    """
    n_products = len(instance.products)
    n_locations = len(instance.locations)
    
    # Calculate location loads
    location_loads = np.zeros(n_locations)
    for i in range(n_products):
        loc_idx = int(assignment[i]) % n_locations
        location_loads[loc_idx] += instance.products[i].space_req
    
    # Check capacity constraints
    for j in range(n_locations):
        if location_loads[j] > instance.locations[j].capacity:
            return False
    
    return True

def repair_assignment(instance: SLAPInstance, assignment: np.ndarray) -> np.ndarray:
    """
    Repair infeasible assignment to satisfy capacity constraints.
    """
    n_products = len(instance.products)
    n_locations = len(instance.locations)
    repaired = assignment.copy()
    
    # Calculate location loads
    location_loads = np.zeros(n_locations)
    for i in range(n_products):
        loc_idx = int(repaired[i]) % n_locations
        location_loads[loc_idx] += instance.products[i].space_req
    
    # Repair overloaded locations
    for j in range(n_locations):
        while location_loads[j] > instance.locations[j].capacity:
            # Find a product to move
            for i in range(n_products):
                if int(repaired[i]) % n_locations == j:
                    # Find alternative location
                    for alt_loc in range(n_locations):
                        if alt_loc != j and location_loads[alt_loc] + instance.products[i].space_req <= instance.locations[alt_loc].capacity:
                            location_loads[j] -= instance.products[i].space_req
                            location_loads[alt_loc] += instance.products[i].space_req
                            repaired[i] = alt_loc
                            break
                    break
    
    return repaired

def initialize_moth_population(instance: SLAPInstance, population_size: int) -> List[Moth]:
    """
    Initialize random moth population with feasible assignments.
    """
    n_products = len(instance.products)
    n_locations = len(instance.locations)
    moths = []
    
    for _ in range(population_size):
        # Generate random assignment
        assignment = np.random.uniform(0, n_locations, n_products)
        
        # Ensure feasibility
        assignment = repair_assignment(instance, assignment)
        
        # Calculate fitness
        cost = calculate_assignment_cost(instance, assignment)
        fitness = 1.0 / (1.0 + cost)  # Higher fitness for lower cost
        
        moths.append(Moth(position=assignment, fitness=fitness, cost=cost))
    
    return moths

In [ ]:
def spiral_flight(moth: Moth, flame: Moth, iteration: int, max_iterations: int, 
                 b: float = 1.0) -> np.ndarray:
    """
    Calculate spiral flight path for moth toward flame.
    Uses logarithmic spiral: D = |Flame - Moth| * exp(b*t) * cos(2πt)
    """
    # Distance between moth and flame
    distance = np.abs(flame.position - moth.position)
    
    # Spiral parameter t decreases linearly from -1 to 1
    t = -1 + (2 * iteration / max_iterations)
    
    # Logarithmic spiral flight
    spiral_factor = np.exp(b * t) * np.cos(2 * np.pi * t)
    
    # New position
    new_position = distance * spiral_factor + flame.position
    
    return new_position

def moth_flame_optimization(instance: SLAPInstance, population_size: int = 20, 
                          max_iterations: int = 150, b: float = 1.0) -> Tuple[Moth, List[float], List[Moth]]:
    """
    Moth-Flame Optimization algorithm for SLAP.
    Returns best solution, convergence history, and final population.
    """
    print(f"Starting MFO: {population_size} moths, {max_iterations} iterations")
    
    # Initialize population
    moths = initialize_moth_population(instance, population_size)
    convergence_history = []
    best_moth_ever = None
    
    for iteration in range(max_iterations):
        # Sort moths by fitness (descending)
        moths.sort(key=lambda m: m.fitness, reverse=True)
        
        # Update best solution
        if best_moth_ever is None or moths[0].fitness > best_moth_ever.fitness:
            best_moth_ever = Moth(
                position=moths[0].position.copy(),
                fitness=moths[0].fitness,
                cost=moths[0].cost
            )
        
        convergence_history.append(best_moth_ever.cost)
        
        # Adaptive number of flames
        num_flames = max(1, round(population_size - iteration * (population_size - 1) / max_iterations))
        
        # Select flames (best moths)
        flames = moths[:num_flames]
        
        # Update moth positions
        new_moths = []
        for i, moth in enumerate(moths):
            # Select flame (each moth follows a different flame)
            flame_idx = min(i, num_flames - 1)
            flame = flames[flame_idx]
            
            # Spiral flight toward flame
            new_position = spiral_flight(moth, flame, iteration, max_iterations, b)
            
            # Ensure feasibility
            new_position = repair_assignment(instance, new_position)
            
            # Calculate fitness
            cost = calculate_assignment_cost(instance, new_position)
            fitness = 1.0 / (1.0 + cost)
            
            new_moths.append(Moth(position=new_position, fitness=fitness, cost=cost))
        
        moths = new_moths
        
        # Progress reporting
        if (iteration + 1) % 30 == 0:
            print(f"Iteration {iteration + 1}: Best Cost = {best_moth_ever.cost:.2f}")
    
    print(f"MFO completed: Final Best Cost = {best_moth_ever.cost:.2f}")
    return best_moth_ever, convergence_history, moths

def convert_assignment_to_discrete(instance: SLAPInstance, position: np.ndarray) -> np.ndarray:
    """
    Convert continuous position to discrete assignment.
    """
    n_products = len(instance.products)
    n_locations = len(instance.locations)
    
    discrete_assignment = np.zeros(n_products, dtype=int)
    for i in range(n_products):
        discrete_assignment[i] = int(position[i]) % n_locations
    
    return discrete_assignment

In [ ]:
# Create the concrete example from the source
def create_concrete_example():
    """Create the example with 6 products and 4 locations"""
    # Products with velocities: [45, 30, 25, 18, 12, 8]
    products = [
        Product(id=0, velocity=45, space_req=1.0, category=1, name="Product 1"),
        Product(id=1, velocity=30, space_req=1.0, category=1, name="Product 2"),
        Product(id=2, velocity=25, space_req=1.0, category=2, name="Product 3"),
        Product(id=3, velocity=18, space_req=1.0, category=2, name="Product 4"),
        Product(id=4, velocity=12, space_req=1.0, category=3, name="Product 5"),
        Product(id=5, velocity=8, space_req=1.0, category=3, name="Product 6")
    ]
    
    # Locations with distances: [2.0, 4.5, 6.0, 8.5]
    locations = [
        Location(id='A', distance=2.0, capacity=3.0, zone=1),  # Can hold up to 3 products
        Location(id='B', distance=4.5, capacity=2.0, zone=1),  # Can hold up to 2 products
        Location(id='C', distance=6.0, capacity=2.0, zone=2),  # Can hold up to 2 products
        Location(id='D', distance=8.5, capacity=2.0, zone=2)   # Can hold up to 2 products
    ]
    
    # Affinity matrix (products in same category have high affinity)
    affinity_matrix = np.array([
        [0, 8, 2, 1, 1, 0],  # Product 1 affinities
        [8, 0, 2, 1, 1, 0],  # Product 2 affinities
        [2, 2, 0, 7, 1, 1],  # Product 3 affinities
        [1, 1, 7, 0, 1, 1],  # Product 4 affinities
        [1, 1, 1, 1, 0, 6],  # Product 5 affinities
        [0, 0, 1, 1, 6, 0]   # Product 6 affinities
    ])
    
    return SLAPInstance(
        products=products,
        locations=locations,
        affinity_matrix=affinity_matrix,
        max_distance=2.0
    )

# Create and solve the instance
instance = create_concrete_example()
best_moth, convergence_history, final_population = moth_flame_optimization(
    instance, population_size=20, max_iterations=150
)

# Convert to discrete assignment
final_assignment = convert_assignment_to_discrete(instance, best_moth.position)

print("\n=== MFO SLAP Results ===")
print(f"\nBest Cost: ${best_moth.cost:.2f}")
print(f"Best Fitness: {best_moth.fitness:.6f}")
print(f"\nFinal Assignment:")
for i, loc_idx in enumerate(final_assignment):
    print(f"  Product {i+1} ({instance.products[i].name}) → Location {instance.locations[loc_idx].id}")

# Expected assignment: [0, 0, 1, 1, 2, 2]
expected_assignment = np.array([0, 0, 1, 1, 2, 2])
expected_cost = calculate_assignment_cost(instance, expected_assignment)
print(f"\nExpected Assignment: {expected_assignment}")
print(f"Expected Cost: ${expected_cost:.2f}")
print(f"Actual vs Expected Cost Difference: ${abs(best_moth.cost - expected_cost):.2f}")

In [ ]:
# Visualization of MFO results
def visualize_mfo_results(instance: SLAPInstance, best_moth: Moth, 
                        convergence_history: List[float], final_assignment: np.ndarray):
    """Create comprehensive visualization of MFO solution"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Moth-Flame Optimization - SLAP Solution Analysis', fontsize=16, fontweight='bold')
    
    # 1. Convergence History
    ax1 = axes[0, 0]
    iterations = range(1, len(convergence_history) + 1)
    ax1.plot(iterations, convergence_history, 'b-', linewidth=2, alpha=0.7)
    ax1.scatter(iterations[::20], convergence_history[::20], c='red', s=30, alpha=0.8)
    ax1.set_title('Convergence History')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Best Cost ($)')
    ax1.grid(True, alpha=0.3)
    ax1.invert_yaxis()  # Lower cost is better
    
    # 2. Final Assignment Heatmap
    ax2 = axes[0, 1]
    assignment_matrix = np.zeros((len(instance.products), len(instance.locations)))
    for i, loc_idx in enumerate(final_assignment):
        assignment_matrix[i, loc_idx] = 1
    
    assignment_df = pd.DataFrame(
        assignment_matrix,
        index=[f"P{i+1}" for i in range(len(instance.products))],
        columns=[f"L{loc.id}" for loc in instance.locations]
    )
    sns.heatmap(assignment_df, annot=True, fmt='d', cmap='Blues', ax=ax2)
    ax2.set_title('Optimal Assignment Matrix')
    ax2.set_xlabel('Storage Locations')
    ax2.set_ylabel('Products')
    
    # 3. Cost Breakdown by Product
    ax3 = axes[1, 0]
    product_costs = []
    product_names = []
    
    for i in range(len(instance.products)):
        loc_idx = final_assignment[i]
        cost = instance.locations[loc_idx].distance * instance.products[i].velocity
        product_costs.append(cost)
        product_names.append(f"P{i+1}")
    
    colors = plt.cm.Set3(np.linspace(0, 1, len(product_costs)))
    bars = ax3.bar(product_names, product_costs, color=colors)
    ax3.set_title('Cost Breakdown by Product')
    ax3.set_xlabel('Product')
    ax3.set_ylabel('Assignment Cost ($)')
    ax3.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, cost in zip(bars, product_costs):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + max(product_costs)*0.01,
                f'${cost:.0f}', ha='center', va='bottom')
    
    # 4. Location Utilization
    ax4 = axes[1, 1]
    location_loads = np.zeros(len(instance.locations))
    for i, loc_idx in enumerate(final_assignment):
        location_loads[loc_idx] += instance.products[i].space_req
    
    capacities = [loc.capacity for loc in instance.locations]
    utilizations = [(location_loads[i] / capacities[i]) * 100 for i in range(len(instance.locations))]
    
    x_pos = np.arange(len(instance.locations))
    bars = ax4.bar(x_pos, utilizations, color=['skyblue', 'lightgreen', 'salmon', 'gold'])
    ax4.set_title('Location Utilization')
    ax4.set_xlabel('Storage Location')
    ax4.set_ylabel('Utilization (%)')
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels([f'L{loc.id}' for loc in instance.locations])
    ax4.grid(True, alpha=0.3)
    
    # Add utilization percentage labels
    for bar, util in zip(bars, utilizations):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{util:.1f}%', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed summary
    print(f"\n{'='*60}")
    print(f"MFO ALGORITHM SUMMARY")
    print(f"{'='*60}")
    print(f"Final Best Cost: ${best_moth.cost:.2f}")
    print(f"Convergence: {convergence_history[0]:.2f} → {convergence_history[-1]:.2f}")
    print(f"Improvement: {((convergence_history[0] - convergence_history[-1]) / convergence_history[0]) * 100:.1f}%")
    
    print(f"\nFinal Assignment Details:")
    for i, loc_idx in enumerate(final_assignment):
        product = instance.products[i]
        location = instance.locations[loc_idx]
        cost = location.distance * product.velocity
        print(f"  {product.name} → Location {location.id} (Cost: ${cost:.2f}, Category: {product.category})")

# Visualize the solution
visualize_mfo_results(instance, best_moth, convergence_history, final_assignment)

In [ ]:
# Performance comparison with baselines
def performance_comparison_mfo(instance: SLAPInstance, mfo_result: Tuple):
    """Compare MFO performance with baseline methods"""
    
    print("\n" + "="*70)
    print("PERFORMANCE COMPARISON - MFO VS BASELINES")
    print("="*70)
    
    best_moth, convergence_history, final_population = mfo_result
    mfo_cost = best_moth.cost
    
    # Baseline 1: Random Assignment
    def random_baseline_trials(num_trials=1000):
        costs = []
        for _ in range(num_trials):
            assignment = np.random.randint(0, len(instance.locations), len(instance.products))
            # Ensure feasibility
            assignment = repair_assignment(instance, assignment.astype(float))
            cost = calculate_assignment_cost(instance, assignment)
            costs.append(cost)
        return np.mean(costs), np.min(costs), np.max(costs)
    
    # Baseline 2: Greedy Assignment (by velocity)
    def greedy_assignment():
        n_products = len(instance.products)
        n_locations = len(instance.locations)
        
        # Sort products by velocity (descending)
        sorted_indices = sorted(range(n_products), key=lambda i: instance.products[i].velocity, reverse=True)
        
        assignment = np.zeros(n_products)
        location_loads = np.zeros(n_locations)
        
        for prod_idx in sorted_indices:
            # Find best available location
            best_loc = None
            best_cost = float('inf')
            
            for loc_idx in range(n_locations):
                if location_loads[loc_idx] + instance.products[prod_idx].space_req <= instance.locations[loc_idx].capacity:
                    cost = instance.locations[loc_idx].distance * instance.products[prod_idx].velocity
                    if cost < best_cost:
                        best_cost = cost
                        best_loc = loc_idx
            
            if best_loc is not None:
                assignment[prod_idx] = best_loc
                location_loads[best_loc] += instance.products[prod_idx].space_req
        
        return calculate_assignment_cost(instance, assignment)
    
    # Baseline 3: Savings Algorithm (simplified)
    def savings_algorithm_baseline():
        # Individual optimal assignment first
        n_products = len(instance.products)
        n_locations = len(instance.locations)
        
        assignment = np.zeros(n_products)
        for i in range(n_products):
            # Find individually optimal location
            best_loc = min(range(n_locations), 
                         key=lambda j: instance.locations[j].distance * instance.products[i].velocity)
            assignment[i] = best_loc
        
        return calculate_assignment_cost(instance, assignment)
    
    # Run baselines
    random_mean, random_min, random_max = random_baseline_trials()
    greedy_cost = greedy_assignment()
    savings_cost = savings_algorithm_baseline()
    
    # Performance comparison table
    print(f"\nAlgorithm Performance Comparison:")
    print(f"{'Method':<20} {'Cost ($)':<12} {'vs Random':<12} {'vs Greedy':<12} {'vs Savings':<12}")
    print("-" * 68)
    
    methods = [
        ("Random (Avg)", random_mean),
        ("Random (Best)", random_min),
        ("Greedy", greedy_cost),
        ("Savings", savings_cost),
        ("MFO", mfo_cost)
    ]
    
    for method_name, cost in methods:
        vs_random = ((random_mean - cost) / random_mean) * 100
        vs_greedy = ((greedy_cost - cost) / greedy_cost) * 100
        vs_savings = ((savings_cost - cost) / savings_cost) * 100
        
        print(f"{method_name:<20} {cost:<12.2f} {vs_random:<12.1f}% {vs_greedy:<12.1f}% {vs_savings:<12.1f}%")
    
    # Expected improvements from source
    print(f"\nExpected vs Actual Improvements:")
    print(f"- Expected improvement over random: 23%")
    print(f"- Actual improvement over random: {((random_mean - mfo_cost) / random_mean) * 100:.1f}%")
    print(f"- Expected improvement over greedy: 8%")
    print(f"- Actual improvement over greedy: {((greedy_cost - mfo_cost) / greedy_cost) * 100:.1f}%")
    
    # Visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle('MFO Performance Comparison', fontsize=16, fontweight='bold')
    
    # Cost comparison
    method_names = [m[0] for m in methods]
    costs = [m[1] for m in methods]
    colors = ['red', 'orange', 'yellow', 'lightgreen', 'green']
    
    bars = ax1.bar(method_names, costs, color=colors, alpha=0.7)
    ax1.set_title('Algorithm Cost Comparison')
    ax1.set_ylabel('Total Cost ($)')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3)
    
    # Add value labels
    for bar, cost in zip(bars, costs):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + max(costs)*0.01,
                f'${cost:.0f}', ha='center', va='bottom')
    
    # Improvement percentage
    improvements = [
        0,  # Random baseline
        0,  # Random best
        ((random_mean - greedy_cost) / random_mean) * 100,
        ((random_mean - savings_cost) / random_mean) * 100,
        ((random_mean - mfo_cost) / random_mean) * 100
    ]
    
    bars2 = ax2.bar(method_names, improvements, color=colors, alpha=0.7)
    ax2.set_title('Improvement Over Random Assignment')
    ax2.set_ylabel('Cost Reduction (%)')
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, alpha=0.3)
    
    # Add percentage labels
    for bar, improvement in zip(bars2, improvements):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + max(improvements)*0.01,
                f'{improvement:.1f}%', ha='center', va='bottom' if improvement >= 0 else 'top')
    
    plt.tight_layout()
    plt.show()

# Run performance comparison
performance_comparison_mfo(instance, (best_moth, convergence_history, final_population))

In [ ]:
# Spiral flight visualization
def visualize_spiral_flight():
    """Visualize the spiral flight pattern of moths"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Moth-Flame Optimization - Spiral Flight Patterns', fontsize=16, fontweight='bold')
    
    # Parameters for spiral visualization
    max_iterations = 150
    b_values = [0.5, 1.0, 1.5, 2.0]
    
    for idx, (ax, b) in enumerate(zip(axes.flat, b_values)):
        # Create sample moth and flame
        moth_pos = np.array([5.0, 5.0])
        flame_pos = np.array([10.0, 10.0])
        
        # Track spiral path
        positions = [moth_pos.copy()]
        
        for iteration in range(max_iterations):
            # Calculate spiral flight
            distance = np.abs(flame_pos - moth_pos)
            t = -1 + (2 * iteration / max_iterations)
            spiral_factor = np.exp(b * t) * np.cos(2 * np.pi * t)
            new_pos = distance * spiral_factor + flame_pos
            positions.append(new_pos.copy())
            moth_pos = new_pos
        
        positions = np.array(positions)
        
        # Plot spiral path
        ax.plot(positions[:, 0], positions[:, 1], 'b-', alpha=0.6, linewidth=1)
        ax.scatter(positions[0, 0], positions[0, 1], c='green', s=100, marker='o', label='Start')
        ax.scatter(positions[-1, 0], positions[-1, 1], c='red', s=100, marker='s', label='End')
        ax.scatter(flame_pos[0], flame_pos[1], c='orange', s=150, marker='*', label='Flame')
        
            
        ax.set_title(f'Spiral Flight (b={b})')
        ax.set_xlabel('X Position')
        ax.set_ylabel('Y Position')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()
    
    print("\nSpiral Flight Pattern Analysis:")
    print("- b parameter controls spiral tightness")
    print("- Lower b values create wider spirals (more exploration)")
    print("- Higher b values create tighter spirals (more exploitation)")
    print("- t parameter decreases from -1 to 1, controlling convergence")

# Visualize spiral flight patterns
visualize_spiral_flight()

### Why this Tier exists vs earlier Tiers
Moth-Flame Optimization provides significant advantages over previous approaches:

**Advantages over Tier 1 (Mathematical Formulation):**
- **Scalability**: Handles much larger instances (100+ products vs limited by exact methods)
- **Global Search**: Population-based search avoids local optima traps
- **Adaptive Mechanism**: Dynamic flame reduction balances exploration/exploitation
- **Parallel Processing**: Multiple moths search simultaneously

**Advantages over Tier 2 (Savings Algorithm):**
- **Global Optimization**: Population-based search vs greedy local decisions
- **Better Solutions**: Typically achieves 8-15% better solutions than greedy methods
- **Diversity**: Multiple candidate solutions prevent premature convergence
- **Flexibility**: Can handle complex constraints and objectives

**Key Innovations:**
- **Spiral Flight Mechanism**: Logarithmic spirals balance exploration and exploitation
- **Adaptive Flames**: Dynamic reduction of guide solutions forces convergence
- **Population Intelligence**: Collective search behavior emerges from individual moth movements
- **Natural Metaphor**: Biologically inspired optimization with proven convergence properties

### When to use this Tier
- **Large-scale warehouses** where exact methods are infeasible
- **Complex constraints** that are difficult for greedy methods
- **Quality-critical applications** where solution optimality matters
- **Research and development** of advanced optimization methods
- **Benchmarking** other metaheuristic algorithms

### Algorithm Characteristics
- **Time Complexity**: O(population_size × iterations × problem_size)
- **Space Complexity**: O(population_size × problem_size)
- **Convergence**: Guaranteed to converge with appropriate parameters
- **Parameters**: Population size, iterations, spiral coefficient (b)
- **Robustness**: Insensitive to initial conditions due to population diversity